## Project layout

Resolve the repo subdirectories so the rest of the notebook is
cwd-independent and figures land in the shared `figures/` folder.


In [ ]:
# Project layout: notebooks live in <root>/notebooks/, source in
# <root>/src/, persistent datasets in <root>/data/, saved figures in
# <root>/figures/. We resolve these once so the rest of the notebook is
# cwd-independent.
from pathlib import Path
import sys

REPO_ROOT   = Path.cwd().resolve().parent
SRC_DIR     = REPO_ROOT / 'src'
DATA_DIR    = REPO_ROOT / 'data'
FIGURES_DIR = REPO_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# Main parameter sweep

Sweeps the four primary parameters at the default geometry:

- `overpressure_mpa` — basal excess pressure
- `k_mud` — mud-cap permeability
- `k_sand` — effective vertical bulk permeability of the column between cap and source
- `ovp_thickness` — thickness of the basal source layer

Geometry is held fixed at NEMP-like defaults. See `02_geometry_sensitivity.ipynb`
for sensitivity to mud-body width, depth-to-source, lateral extent of overpressure,
domain width, and the presence of an intermediate seal.

In [ ]:
import itertools
import time
import warnings

import h5py
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from pockmark_model import PERM_FACTOR, run_simulation
from sweep_utils import (
    build_sample_cols, fmt_duration, load_done_keys,
    run_key_str, write_run_to_hdf5,
)

warnings.filterwarnings('ignore')

In [ ]:
# ── Sweep axes (k values are stored internally in m/yr; m² values are
# obtained by dividing by PERM_FACTOR before writing to disk) ───────────
overpressure_mpa_values = list(np.linspace(0.7, 2.4, 9))
k_mud_values = np.unique([
    k * PERM_FACTOR for k in np.concatenate([
        np.logspace(-14, -18, 5),
        np.logspace(-18, -20, 9),
    ])
])
k_sand_values = [k * PERM_FACTOR for k in np.logspace(-9, -13, 5)]
ovp_thickness_values = list(np.linspace(1, 150, 11))

# Fixed geometry / non-swept parameters for the main sweep
FIXED = dict(
    Lx                 = 105_000.0,
    mud_width          = 35_000.0,
    depth_to_source    = 300.0,
    ovp_lateral_extent = None,
    seal_thickness     = 0.0,
    ss                 = 1e-4,
    sy                 = 0.15,
    bulkrho_mud        = 1650.0,
    bulkrho_sand       = 2150.0,
)

HDF5_FILE = str(DATA_DIR / 'groundwater_sweep_profiles.h5')

runs = list(itertools.product(
    overpressure_mpa_values, k_mud_values, k_sand_values, ovp_thickness_values,
))
total = len(runs)
print(f'Planning {total} runs.')

In [ ]:
def _key(op, km, ks, ot):
    return run_key_str(
        overpressure_mpa = op,
        k_mud_m2         = km / PERM_FACTOR,
        k_sand_m2        = ks / PERM_FACTOR,
        ovp_thickness_m  = ot,
    )

def _run_one(op, km, ks, ot):
    try:
        res, _cm, _cs, grad_sp, x_c, in_mud = run_simulation(
            overpressure_mpa=op, k_mud=km, k_sand=ks, ovp_thickness=ot, **FIXED,
        )
        return (op, km, ks, ot), res, grad_sp, x_c, in_mud, None
    except Exception as e:
        return (op, km, ks, ot), None, None, None, None, str(e)

done_keys = load_done_keys(HDF5_FILE)
runs_todo = [r for r in runs if _key(*r) not in done_keys]
print(f'{len(done_keys)} already done, {len(runs_todo)} remaining.')

In [ ]:
failed_runs           = {}
sweep_results_summary = {}
_sample_cols          = None
start_time            = time.time()
already_done          = total - len(runs_todo)

if runs_todo:
    with h5py.File(HDF5_FILE, 'a') as hf:
        if 'runs' not in hf:
            hf.attrs['description']  = 'Main parameter sweep — gradient profiles'
            hf.attrs['rho_seawater'] = 1024.0
            hf.create_group('runs')

        for batch_done, result in enumerate(
            Parallel(n_jobs=8, backend='loky', verbose=0, return_as='generator')(
                delayed(_run_one)(*r) for r in runs_todo
            ), start=1
        ):
            (op, km, ks, ot), res, grad_sp, x_c, in_mud, err = result
            if err is not None:
                failed_runs[(op, km, ks, ot)] = err
                continue

            # Initialise shared coord/mask datasets from the first run
            if _sample_cols is None:
                _sample_cols = build_sample_cols(x_c)
                if 'x_coords_m'  not in hf:
                    hf.create_dataset('x_coords_m',  data=x_c[_sample_cols].astype(np.float32), compression='gzip')
                if 'in_mud_zone' not in hf:
                    hf.create_dataset('in_mud_zone', data=in_mud[_sample_cols],                  compression='gzip')

            params = {
                'overpressure_mpa': op,
                'k_mud_m2':         km / PERM_FACTOR,
                'k_sand_m2':        ks / PERM_FACTOR,
                'ovp_thickness_m':  ot,
            }
            write_run_to_hdf5(
                hf, _key(op, km, ks, ot), params,
                res['time'].values,
                np.asarray(grad_sp)[:, _sample_cols],
            )
            sweep_results_summary[(op, km, ks, ot)] = res

            if batch_done % 10 == 0 or batch_done == len(runs_todo):
                elapsed = time.time() - start_time
                rate    = batch_done / elapsed if elapsed > 0 else 0
                eta     = (len(runs_todo) - batch_done) / rate if rate > 0 else float('inf')
                eta_str = fmt_duration(eta) if eta < 1e8 else '…'
                print(f'  {already_done + batch_done}/{total}  elapsed {fmt_duration(elapsed)}  ETA {eta_str}')
    print(f'Done. Successful: {len(sweep_results_summary)}, failed: {len(failed_runs)}')
else:
    print('All runs already complete.')

## Build summary CSV

One row per (model run × ρ_mud × ρ_sand). Density combinations are
post-processed from the stored gradient profiles, so they don't require
additional model runs.

In [ ]:
RHO_SW              = 1024.0
bulkrho_mud_values  = [1300, 1350, 1400, 1450, 1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850, 1900]
bulkrho_sand_values = [1800, 1850, 1900, 1950, 2000, 2050, 2100, 2150, 2200]

# If we just ran the sweep, sweep_results_summary is populated; otherwise
# reconstruct from the stored gradient profiles.
if not sweep_results_summary:
    sweep_results_summary = {}
    with h5py.File(HDF5_FILE, 'r') as hf:
        in_mud = hf['in_mud_zone'][:]
        for grp_name in hf['runs']:
            g = hf['runs'][grp_name]
            key = (
                float(g.attrs['overpressure_mpa']),
                float(g.attrs['k_mud_m2']),
                float(g.attrs['k_sand_m2']),
                float(g.attrs['ovp_thickness_m']),
            )
            times = g['time_yr'][:]
            grads = g['gradients'][:].astype(np.float32)
            sweep_results_summary[key] = pd.DataFrame({
                'time':                 times,
                'max_gradient_mud':     np.where(in_mud,  grads, 0.0).max(axis=1),
                'max_gradient_sand':    np.where(~in_mud, grads, 0.0).max(axis=1),
                'overall_max_gradient': grads.max(axis=1),
            })

def _exceed_stats(grad_ts: np.ndarray, crit: float, times: np.ndarray) -> dict:
    mask = grad_ts > crit
    t_exc = times[mask]
    return {
        'any_exceed':       bool(mask.any()),
        'first_exceed_yr':  float(t_exc[0])  if len(t_exc) else float('nan'),
        'last_exceed_yr':   float(t_exc[-1]) if len(t_exc) else float('nan'),
        'n_timesteps_exceed': int(mask.sum()),
    }

# When iterating the sweep keys: op is in MPa, km/ks are in m/yr.
# We store the m² form in the CSV (matches the HDF5 attrs).
rows = []
for (op, km_yr, ks_yr, ot), res in sweep_results_summary.items():
    times    = res['time'].values
    g_mud    = res['max_gradient_mud'].values
    g_sand   = res['max_gradient_sand'].values
    base_row = {
        'overpressure_mpa': op,
        'k_mud_m2':  km_yr / PERM_FACTOR,
        'k_sand_m2': ks_yr / PERM_FACTOR,
        'ovp_thickness_m':       ot,
        'peak_gradient_mud':     float(g_mud.max()),
        'peak_gradient_sand':    float(g_sand.max()),
        'peak_gradient_overall': float(res['overall_max_gradient'].max()),
        'n_timesteps':           len(times),
        'time_start_yr':         float(times[0]),
        'time_end_yr':           float(times[-1]),
    }
    for rho_m in bulkrho_mud_values:
        cm = (rho_m - RHO_SW) / RHO_SW
        m_stats = _exceed_stats(g_mud, cm, times)
        for rho_s in bulkrho_sand_values:
            cs = (rho_s - RHO_SW) / RHO_SW
            s_stats = _exceed_stats(g_sand, cs, times)
            rows.append({
                **base_row,
                'bulkrho_mud_kg_m3':  rho_m,
                'bulkrho_sand_kg_m3': rho_s,
                'crit_grad_mud':      cm,
                'crit_grad_sand':     cs,
                'any_exceed_mud':         m_stats['any_exceed'],
                'first_exceed_yr_mud':    m_stats['first_exceed_yr'],
                'last_exceed_yr_mud':     m_stats['last_exceed_yr'],
                'n_timesteps_exceed_mud': m_stats['n_timesteps_exceed'],
                'any_exceed_sand':         s_stats['any_exceed'],
                'first_exceed_yr_sand':    s_stats['first_exceed_yr'],
                'last_exceed_yr_sand':     s_stats['last_exceed_yr'],
                'n_timesteps_exceed_sand': s_stats['n_timesteps_exceed'],
            })
df_summary = pd.DataFrame(rows)
df_summary.to_csv(DATA_DIR / 'groundwater_sweep_summary.csv', index=False)
print(f'Wrote summary: {len(df_summary):,} rows.')